In [ ]:
# ED_Phase 2_scores_v1
# UI panel integration for Phase-2 risk scores, behind RUN_UI guard

if CONFIG.get("RUN_UI", False):
    import streamlit as st

    with st.expander("Phase-2 Risk Scores", expanded=True):
        fd = s.feature_dict() if 's' in locals() else {}
        rs = fd.get("risk_scores") if CONFIG.get("RUN_PIPELINE", False) else None

        if rs is None:
            st.info("Risk scores not computed (enable RUN_PIPELINE).")
        else:
            st.subheader("Computed Scores")
            cols = st.columns(3)
            simple_keys = ["HEART", "GRACE_inhosp_points", "Marburg", "qSOFA", "MEWS"]
            for i,k in enumerate(simple_keys):
                cols[i%3].metric(k, rs.get(k))

            st.markdown("---")
            st.subheader("SOFA Breakdown")
            sofa = rs.get("SOFA", {})
            if isinstance(sofa, dict):
                for sub, val in sofa.items():
                    st.write(f"{sub}: {val}")

            st.markdown("---")
            st.subheader("D-Dimer Thresholds")
            d = rs.get("D_Dimer", {})
            st.write("Age-adjusted (ug/L FEU):", d.get("age_adjusted_threshold_ug_L_FEU"))
            st.write("Pregnancy YEARS threshold (ng/mL FEU):", d.get("pregnancy_threshold_years_ng_mL_FEU"))

            # adjustable thresholds
            st.markdown("---")
            st.subheader("Threshold Controls")
            dd_cutoff = st.number_input("D-Dimer alert cutoff (ug/L FEU)", value=float(d.get("age_adjusted_threshold_ug_L_FEU") or 500))
            if dd_cutoff and d.get("age_adjusted_threshold_ug_L_FEU") and d["age_adjusted_threshold_ug_L_FEU"] > dd_cutoff:
                st.error("Age-adjusted D-Dimer above cutoff")

            mews_alert = st.slider("MEWS alert threshold", 0, 15, 5)
            if rs.get("MEWS") and rs["MEWS"] >= mews_alert:
                st.error(f"MEWS exceeds threshold: {rs['MEWS']}")

            qsofa_alert = st.slider("qSOFA alert threshold", 0, 3, 2)
            if rs.get("qSOFA") and rs["qSOFA"] >= qsofa_alert:
                st.error(f"qSOFA exceeds threshold: {rs['qSOFA']}")

            heart_alert = st.slider("HEART alert threshold", 0, 10, 7)
            if rs.get("HEART") and rs["HEART"] >= heart_alert:
                st.error(f"HEART exceeds threshold: {rs['HEART']}")

            grace_alert = st.slider("GRACE alert threshold (points)", 0, 400, 150)
            if rs.get("GRACE_inhosp_points") and rs["GRACE_inhosp_points"] >= grace_alert:
                st.error(f"GRACE points exceed threshold: {rs['GRACE_inhosp_points']}")
